In [ ]:
import pandas as pd


def study_scaffold_concentration(dataset) -> pd.DataFrame:
    """Analyse par tache : taille du dataset annote, desequilibre et
    concentration des positifs par famille de scaffold.

    Retourne un DataFrame (12 lignes) :
      task, n_annotated, n_pos, pos_rate, n_scaffolds_pos,
      top3_share (part des positifs dans les 3 plus grosses familles).
    """
    # -- extraction directe depuis le dataset (plus d'argument Y externe) --
    Y = stack_labels(dataset)  # (n_mol, 12), NaN conserves

    # -- familles de scaffolds globales (identiques au split) --
    scaffold_to_idx = {}
    for i in range(len(dataset)):
        scaffold_to_idx.setdefault(murcko_scaffold_smiles(dataset[i]), []).append(i)

    rows = []
    for t, name in enumerate(TASK_NAMES):
        y_t = Y[:, t]
        annotated = ~np.isnan(y_t)
        n_annotated = int(annotated.sum())
        pos_idx = np.where(y_t == 1)[0]  # indices des positifs

        if len(pos_idx) == 0:  # tache sans aucun positif
            rows.append(
                {
                    "task": name,
                    "n_annotated": n_annotated,
                    "n_pos": 0,
                    "pos_rate": 0.0,
                    "n_scaffolds_pos": 0,
                    "top3_share": np.nan,
                }
            )
            continue

        # familles des molécules positives (une famille peut apparaître plusieurs
        # fois si plusieurs positifs la partagent — c'est justement le signal)
        pos_family_sizes = [
            len(scaffold_to_idx[murcko_scaffold_smiles(dataset[i])]) for i in pos_idx
        ]
        top3_share = float(np.sort(pos_family_sizes)[-3:].sum() / len(pos_idx))

        rows.append(
            {
                "task": name,
                "n_annotated": n_annotated,  # molécules testées sur cette tâche
                "n_annotated_share": 100 * round(n_annotated / 7823, 2),
                "n_pos": len(pos_idx),
                "n_scaffolds_pos": len(
                    set(murcko_scaffold_smiles(dataset[i]) for i in pos_idx)
                ),
                "pos_rate (%) ": round(100 * len(pos_idx) / n_annotated, 2),
                "top3_share (%)": round(top3_share, 2),
            }
        )
    df = pd.DataFrame(rows)
    df["AUC_SVM"] = np.array(
        [0.77, 0.84, 0.85, 0.74, 0.71, 0.81, 0.76, 0.73, 0.8, 0.74, 0.84, 0.77]
    ) - ([0.014 for i in range(12)])
    df["AUC_SVM_Tanimoto"] = [
        0.76,
        0.84,
        0.84,
        0.75,
        0.71,
        0.78,
        0.76,
        0.765,
        0.8,
        0.74,
        0.85,
        0.78,
    ]
    df["AUC_LGBM"] = [
        0.67,
        0.79,
        0.84,
        0.71,
        0.64,
        0.71,
        0.695,
        0.72,
        0.73,
        0.71,
        0.83,
        0.74,
    ]

    print(df.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    return df


def murcko_scaffold_smiles(data) -> str:
    """SMILES canonique du scaffold Bemis-Murcko : la clef de regroupement."""
    mol = Chem.MolFromSmiles(data.smiles)
    if mol is None:
        return ""
    return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))

In [ ]:
stat_EDA = study_scaffold_concentration(load_tox21())

In [ ]:
stat_EDA = stat_EDA.set_index("task")

In [ ]:
def plot_correlation_heatmap(
    df: pd.DataFrame,
    title: str = "Matrice de corrélation",
    out: Path = FIG_DIR / "correlation_heatmap.png",
):
    """Heatmap de correlation (RdBu), divergente autour de 0.
    Rouge = correlation positive, bleu = negative (centree sur zero)."""
    corr = df.corr(numeric_only=True)

    fig, ax = plt.subplots(
        figsize=(2 + 0.6 * len(corr.columns), 1.5 + 0.5 * len(corr.columns))
    )
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)

    # etiquettes : noms de variables sur les axes
    labels = corr.columns
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels)), labels)

    # annotations : valeur dans chaque cellule
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(
                j,
                i,
                f"{corr.iloc[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=8,
                color="white" if abs(corr.iloc[i, j]) > 0.5 else "black",
            )

    fig.colorbar(im, ax=ax, label="Corrélation de Pearson", shrink=0.8)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Figure sauvegardée : {out}")
    return fig


plot_correlation_heatmap(stat_EDA)

In [ ]:
stat_EDA.head(12)

In [ ]:
# ------------------------------------------------------------------ scaffold split
def scaffold_split(dataset, frac_train=0.8, frac_val=0.1, frac_test=0.1, seed: int = 0):
    """Scaffold split : toutes les molecules d'un meme scaffold dans le MEME split
    -> pas de fuite structurelle (contrairement au split aleatoire).
    Groupes tries par taille decroissante, remplissage train -> val -> test,
    ordre melange par seed -> split deterministe."""
    assert abs(frac_train + frac_val + frac_test - 1.0) < 1e-6
    rng = torch.Generator().manual_seed(seed)

    scaffold_to_idx = {}
    for i in range(len(dataset)):
        scaffold_to_idx.setdefault(murcko_scaffold_smiles(dataset[i]), []).append(i)

    # Le seed doit mélanger l'ordre DES GROUPES (pas leur interieur) :
    # permuter l'interieur d'un groupe ne change pas le split de ses membres,
    # d'ou des results rigoureusement identiques pour tous les seeds (bug fixe).
    groups = sorted(scaffold_to_idx.values(), key=len, reverse=True)
    order = torch.randperm(len(groups), generator=rng).tolist()
    groups = [groups[k] for k in order]

    n = len(dataset)
    train, val, test = [], [], []
    for group in groups:
        for idx in group:
            if len(train) < frac_train * n:
                train.append(idx)
            elif len(val) < frac_val * n:
                val.append(idx)
            else:
                test.append(idx)
    return train, val, test


# ---------------------------------------------------------------- featurisation


def smiles_to_fingerprint(
    smiles: str, radius: int = 2, n_bits: int = 2048
) -> np.ndarray:
    """ECFP4 : SMILES -> vecteur binaire (n_bits,). Sparse, sans fit -> pas de fuite."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"SMILES invalide : {smiles!r}")
    fp = MORGAN_GEN.GetFingerprint(mol)
    return np.asarray(fp, dtype=np.float32)


def build_feature_matrix(dataset) -> np.ndarray:
    """Tout le dataset -> X (n_molecules, n_bits). Une seule fois, reutilise partout."""
    X = np.stack([smiles_to_fingerprint(d.smiles) for d in dataset])
    print(f"Features : X {X.shape}, densite {X.mean():.4f}")
    return X


def stack_labels(dataset) -> np.ndarray:
    """y de PyG (n, 1, 12) -> Y (n, 12) avec NaN conserves."""
    return torch.stack([d.y for d in dataset]).squeeze(1).numpy()


# --------------------------------------------------------------------- entrainement


def train_svm_per_task(X: np.ndarray, Y: np.ndarray, idx_train: list, C: float = 1.0):
    """Un LinearSVC par tache, sur les lignes annotees uniquement.
    class_weight='balanced' compense le desequilibre mesure dans l'EDA.
    Decision_function servira pour l'AUC -> pas de probability=True (Platt couteux)."""
    models = []
    for t in range(Y.shape[1]):
        mask_tr = ~np.isnan(Y[idx_train, t])
        Xtr = X[idx_train][mask_tr]
        ytr = Y[idx_train, t][mask_tr].astype(int)
        clf = LinearSVC(C=C, class_weight="balanced", max_iter=10_000)
        clf.fit(Xtr, ytr)
        models.append(clf)
    return models


# ---------------------------------------------------------------------- evaluation


def evaluate_svm(models, X: np.ndarray, Y: np.ndarray, idx) -> dict:
    """AUC par tache sur les lignes annotees du split, + moyenne.
    AUC = metrique de ranking : les marges signees suffisent."""
    aucs = []
    for t, clf in enumerate(models):
        mask = ~np.isnan(Y[idx, t])
        if mask.sum() < 2 or len(np.unique(Y[idx, t][mask])) < 2:
            aucs.append(np.nan)
            continue
        scores = clf.decision_function(X[idx][mask])
        aucs.append(roc_auc_score(Y[idx, t][mask].astype(int), scores))
    aucs = np.array(aucs)
    return {"auc_per_task": aucs, "auc_mean": np.nanmean(aucs)}


def auc_for_task(clf, X, Y, idx_val, t) -> float:
    mask = ~np.isnan(Y[idx_val, t])
    if mask.sum() < 2 or len(np.unique(Y[idx_val, t][mask])) < 2:
        return np.nan
    return roc_auc_score(
        Y[idx_val, t][mask].astype(int), clf.decision_function(X[idx_val][mask])
    )


def tanimoto_kernel(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """Similarite Tanimoto vectorisee entre deux ensembles de fingerprints.
    K(i,j) = |Ai & Bj| / |Ai | Bj| - |Ai & Bj|   (bits communs / union des bits)"""
    inter = A @ B.T
    norm_a = A.sum(axis=1, keepdims=True)
    norm_b = B.sum(axis=1, keepdims=True).T
    union = norm_a + norm_b - inter
    return inter / union


def train_svm_tanimoto(G_full, Y, idx_train, C: float = 1.0):
    """Le masque NaN s'applique aux lignes ET colonnes via np.ix_."""
    models = []
    for t in range(Y.shape[1]):
        rows = np.array([i for i in idx_train if not np.isnan(Y[i, t])])
        G_tt = G_full[np.ix_(rows, rows)]  # Gram train masquee, symetrique
        clf = SVC(kernel="precomputed", C=C, class_weight="balanced")
        clf.fit(G_tt, Y[rows, t].astype(int))
        models.append(clf)
    return models


def evaluate_svm_tanimoto(models, G_full, Y, idx_train, idx_test) -> dict:
    aucs = []
    for t, clf in enumerate(models):
        rows_tr = np.array([i for i in idx_train if not np.isnan(Y[i, t])])
        rows_te = np.array([i for i in idx_test if not np.isnan(Y[i, t])])
        G_te = G_full[np.ix_(rows_te, rows_tr)]  # (n_test, n_train) — largeur du fit
        scores = clf.decision_function(G_te)
        aucs.append(roc_auc_score(Y[rows_te, t].astype(int), scores))
    aucs = np.array(aucs)
    return {"auc_per_task": aucs, "auc_mean": np.nanmean(aucs)}


# ------------------------------------------------------------------------- pipeline


def plot_auc_per_task(
    model_name: str, auc_matrix: np.ndarray, out: Path = FIG_DIR / "auc_per_task.png"
):
    """Barres horizontales : AUC par tache (moyenne +/- std sur les seeds).
    Sauvegarde dans results/figures (versionne) + retourne la figure."""
    aucs = np.nanmean(auc_matrix, axis=0)
    stds = np.nanstd(auc_matrix, axis=0)
    order = np.argsort(aucs)  # triees de la pire a la meilleure

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(
        np.array(TASK_NAMES)[order],
        aucs[order],
        xerr=stds[order],
        color="#4C72B0",
        alpha=1,
    )
    ax.axvline(0.5, color="red", linestyle="--", linewidth=1, label="Hasard (0.5)")
    ax.axvline(
        np.mean(aucs),
        color="orange",
        linestyle="--",
        linewidth=1,
        label=" Mean of models",
    )
    ax.set_xlim(0.4, 1)
    ax.set_xlabel("AUC ROC (test, scaffold split)")
    ax.set_title(f"Baseline {model_name} + ECFP4 - AUC par tache")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out, dpi=150)
    print(f"Mean  : {np.mean(aucs)}, std : {np.std(aucs)}")
    print(f"Figure sauvegardee : {out}")
    return fig


# ------------------------------------------------------------------------- pipeline

In [ ]:
import lightgbm as lgb


def train_lgbm_per_task(params, X, Y, idx_train) -> list:
    """12 LGBMClassifier, un par tache, sur les lignes annotees uniquement.
    Contrainte LGBM : pas de NaN dans y (on drope), mais X peut rester intact
    (ECFP est binaire, aucun NaN de toute facon).
    scale_pos_weight = n_negatifs / n_positifs : l'equivalent natif LGBM
    de class_weight='balanced', calcule PAR TACHE (le ratio varie)."""
    models = []
    for t in range(Y.shape[1]):
        mask = ~np.isnan(Y[idx_train, t])
        y_t = Y[idx_train, t][mask].astype(int)
        n_pos = y_t.sum()
        n_neg = len(y_t) - n_pos
        spw = n_neg / max(n_pos, 1)  # garde-fou si 0 positif

        clf = lgb.LGBMClassifier(
            reg_lambda=params["L2"],  # Force de la régularisation L2 (valeur > 0)
            reg_alpha=params["L1"],  # Régularisation L1 optionnelle
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            num_leaves=params["num_leaves"],
            scale_pos_weight=spw,
            verbosity=-1,  # silence les logs internes
            random_state=42,  # LGBM est stochastique : fixe-le
        )
        clf.fit(X[idx_train][mask], y_t)
        models.append(clf)
    return models


def evaluate_lgbm_per_task(models, X, Y, idx_test) -> dict:
    """AUC par tache sur lignes annotees, via predict_proba (probabilites
    natives de LGBM — pas besoin du hack decision_function du SVM)."""
    aucs = []
    for t, clf in enumerate(models):
        mask = ~np.isnan(Y[idx_test, t])
        if mask.sum() < 2 or len(np.unique(Y[idx_test, t][mask])) < 2:
            aucs.append(np.nan)
            continue
        proba = clf.predict_proba(X[idx_test][mask])[:, 1]
        aucs.append(roc_auc_score(Y[idx_test, t][mask].astype(int), proba))
    aucs = np.array(aucs)
    return {"auc_per_task": aucs, "auc_mean": np.nanmean(aucs)}

In [ ]:
def run(
    model_name: str = "SVM",
    seed_list=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 20, 45, 67),
    params: dict[float] = {
        "L1": 1,
        "L2": 10,
        "n_estimators": 300,
        "learning_rate": 0.05,
        "num_leaves": 31,
    },
):
    dataset = load_tox21()
    X = build_feature_matrix(dataset)
    G = tanimoto_kernel(X, X)
    Y = stack_labels(dataset)

    all_means, all_aucs = [], []
    for seed in tqdm(seed_list):
        train, val, test = scaffold_split(dataset, seed=seed)
        if model_name == "SVM":
            models = train_svm_tanimoto(params, G, Y, train)
            res = evaluate_svm_tanimoto(models, G, Y, train, test)
        elif model_name == "LGBM":
            m_lgbm = train_lgbm_per_task(params, X, Y, train)
            res = evaluate_lgbm_per_task(m_lgbm, X, Y, test)
        # print(f"seed={seed} | AUC moyenne test = {res['auc_mean']:.4f}")
        # print("  par tache :", np.round(res["auc_per_task"], 3))
        all_means.append(res["auc_mean"])
        all_aucs.append(res["auc_per_task"])

    m = np.array(all_means)
    # print(f"\nSVM lineaire : AUC moyenne = {m.mean():.4f} +/- {m.std():.4f}")

    # Tableau console lisible
    auc_matrix = np.stack(all_aucs)  # (n_seeds, 12)
    # print("\n" + "-" * 50)
    # for name, a, s in zip(TASK_NAMES, np.nanmean(auc_matrix, 0), np.nanstd(auc_matrix, 0)):
    #     print(f"  {name:<14} {a:.3f} +/- {s:.3f}")
    # print("-" * 50)

    plot_auc_per_task(model_name, auc_matrix)
    # np.savetxt(FIG_DIR.parent / "svm_auc_per_task.csv",
    #            auc_matrix, delimiter=",", header=",".join(TASK_NAMES), comments="")
    print("Ecart-type", m.std())
    return m.mean()


# if __name__ == "__main__":
#   run()

In [ ]:
seeds = [i for i in range(100)]
seeds_test = [1, 2, 3, 4, 5]
param_C = 1
run(model_name="LGBM", seed_list=seeds_test)

In [ ]:
def op_score(Hparams: dict) -> float:
    seeds_params = [i for i in range(10)]
    auc_mean = run(model_name="LGBM", seed_list=seeds_params, params=Hparams)
    return auc_mean


HP = {"L1": 0, "L2": 1, "n_estimators": 300, "learning_rate": 0.05, "num_leaves": 31}


def select_HP() -> Tuple[list, list]:
    est = [100, 300, 500, 1000]
    lr = [0.01, 0.05, 0.1]
    nl = [21, 31, 61]
    Res = []
    Val = []
    for x in tqdm(product(est, lr, nl)):
        params_var = {"n_estimators": x[0], "learning_rate": x[1], "num_leaves": x[2]}
        Res.append(op_score(params_var))
        Val.append(x)
    return Res, Val

In [ ]:
Res, Val = select_HP()

In [ ]:
import seaborn as sns

print(Res)

In [2]:
from optimizer import Apollon

C:\Users\Elfamoso\OneDrive\Documents\GitHub\Victorien_Bigarre_Portfolio\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
Apo = Apollon(model_name="SVM_Linear", n_trials=30)

In [4]:
best_params, best_value = Apo.optimizer()

[I 2026-10-10 21:54:11,261] A new study created in memory with name: no-name-50c856a8-e73d-41fb-90bb-13163ff60e13


Tox21 : 7823 molecules


[21:54:11] WARNING: not removing hydrogen atom without neighbors
[21:54:15] WARNING: not removing hydrogen atom without neighbors


Features : X (7823, 2048), densite 0.0146


100%|██████████| 10/10 [01:10<00:00,  7.10s/it]
[I 2026-10-10 21:55:48,030] Trial 0 finished with value: 0.7250525239285155 and parameters: {'C': 0.023522263737794622}. Best is trial 0 with value: 0.7250525239285155.


...

Tox21 : 7823 molecules


[22:41:35] WARNING: not removing hydrogen atom without neighbors
[22:41:44] WARNING: not removing hydrogen atom without neighbors


Features : X (7823, 2048), densite 0.0146


100%|██████████| 10/10 [01:09<00:00,  6.93s/it]
[I 2026-10-10 22:43:15,677] Trial 29 finished with value: 0.7090516119267003 and parameters: {'C': 0.09957404730173912}. Best is trial 9 with value: 0.7316555728122262.


Meilleur AUC val : 0.7317
Meilleurs params : {'C': 0.005699699999623568}


In [5]:
print(f"Le mieux c'est :{best_value}")

Le mieux c'est :0.7316555728122262


In [12]:
from models import Gaia

u = Gaia(
    model_name="SVM_Tanimoto", seed_list=[i for i in range(100)], params={"C": 0.607}
)

Tox21 : 7823 molecules


[23:07:59] WARNING: not removing hydrogen atom without neighbors
[23:08:06] WARNING: not removing hydrogen atom without neighbors


Features : X (7823, 2048), densite 0.0146


100%|██████████| 100/100 [17:29<00:00, 10.49s/it]


In [13]:
print(u.score["test"]["auc_mean"])

0.7860924458730194


In [8]:
print("best_params:", best_params, "best_value:", best_value)

best_params: {'C': 0.005699699999623568} best_value: 0.7316555728122262


In [2]:
from models import Gaia

In [11]:
obj = Gaia(
    model_name="LGBM",
    seed_list=[i for i in range(100)],
    params={
        "num_leaves": 56,
        "n_estimators": 746,
        "learning_rate": 0.0997,
        "L2": 15.21,
        "L1": 0,
    },
)

Tox21 : 7823 molecules


[18:00:41] WARNING: not removing hydrogen atom without neighbors


Features : X (7823, 2048), densite 0.0146


[18:01:25] WARNING: not removing hydrogen atom without neighbors


 n_annotated  n_annotated_share  n_pos  n_scaffolds_pos  pos_rate (%)   top3_share (%)  AUC_SVM  AUC_SVM_Tanimoto  AUC_LGBM
        7258             93.000    308              162          4.240          17.290    0.710             0.730     0.715
        6751             86.000    237              107          3.510          22.470    0.761             0.827     0.811
        6542             84.000    768              358         11.740           6.930    0.756             0.841     0.827
        5815             74.000    300              189          5.160          17.750    0.685             0.776     0.752
        6186             79.000    791              358         12.790           6.730    0.622             0.678     0.675
        6948             89.000    349              156          5.020          15.260    0.694             0.738     0.730
        6443             82.000    186              107          2.890          28.630    0.717             0.779     0.738
        

[18:01:35] WARNING: not removing hydrogen atom without neighbors


Figure sauvegardee : results\figures\scaffold_distribution.png
           n_molecules : 7823
           n_scaffolds : 2404
      mean_family_size : 3.2541597337770383
    median_family_size : 1.0
       max_family_size : 1775
          n_singletons : 1871
           top10_share : 0.47603221270612295


100%|██████████| 100/100 [1:19:46<00:00, 47.86s/it]


In [14]:
import numpy as np

TASK_NAMES = [
    "NR-AR",
    "NR-AR-LBD",
    "NR-AhR",
    "NR-ER",
    "NR-ER-LBD",
    "NR-PPAR-gamma",
    "NR-Aromatase",
    "SR-ARE",
    "SR-ATAD5",
    "SR-HSE",
    "SR-MMP",
    "SR-p53",
]
df = u.score["test"]["auc_matrix"]
a = []
for i, x in enumerate(df.T):
    print(f"{TASK_NAMES[i]}: {x.mean()}")
    a.append(x.mean())
print(a)
print(np.mean(a), np.std(a))

NR-AR: 0.76636379624712
NR-AR-LBD: 0.8508728081174223
NR-AhR: 0.8511887392169402
NR-ER: 0.7576396341109417
NR-ER-LBD: 0.7127062707483469
NR-PPAR-gamma: 0.78081826858475
NR-Aromatase: 0.773847006025392
SR-ARE: 0.7706591668322907
SR-ATAD5: 0.8018759451236958
SR-HSE: 0.7385897030080996
SR-MMP: 0.8483839917277287
SR-p53: 0.7801640207335038
[np.float64(0.76636379624712), np.float64(0.8508728081174223), np.float64(0.8511887392169402), np.float64(0.7576396341109417), np.float64(0.7127062707483469), np.float64(0.78081826858475), np.float64(0.773847006025392), np.float64(0.7706591668322907), np.float64(0.8018759451236958), np.float64(0.7385897030080996), np.float64(0.8483839917277287), np.float64(0.7801640207335038)]
0.7860924458730193 0.04267640032389349


10

In [1]:
from dataset import Data

EDA = Data()

C:\Users\Elfamoso\OneDrive\Documents\GitHub\Victorien_Bigarre_Portfolio\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tox21 : 7823 molecules


[23:33:26] WARNING: not removing hydrogen atom without neighbors
[23:33:32] WARNING: not removing hydrogen atom without neighbors


Features : X (7823, 2048), densite 0.0146


In [2]:
EDA.extract(EDA.EDA_statistique(EDA.df))

[23:33:54] WARNING: not removing hydrogen atom without neighbors


 n_annotated  n_annotated_share  n_pos  n_scaffolds_pos  pos_rate (%)   top3_share (%)  AUC_SVM  AUC_SVM_Tanimoto  AUC_LGBM
        7258             93.000    308              162          4.240          17.290    0.764             0.766     0.715
        6751             86.000    237              107          3.510          22.470    0.835             0.851     0.811
        6542             84.000    768              358         11.740           6.930    0.838             0.851     0.827
        5815             74.000    300              189          5.160          17.750    0.738             0.758     0.752
        6186             79.000    791              358         12.790           6.730    0.704             0.713     0.675
        6948             89.000    349              156          5.020          15.260    0.779             0.781     0.730
        6443             82.000    186              107          2.890          28.630    0.752             0.774     0.738
        

<Figure size 740x600 with 2 Axes>

In [4]:
u.plot_auc_per_task(model_name="SVM_Tanimoto",auc_matrix=df)

NameError: name 'u' is not defined

In [6]:
import pandas as pd

Stats = pd.DataFrame(EDA.EDA_statistique(EDA.df))

[12:37:47] WARNING: not removing hydrogen atom without neighbors


 n_annotated  n_annotated_share  n_pos  n_scaffolds_pos  pos_rate (%)   top3_share (%)  AUC_SVM  AUC_SVM_Tanimoto  AUC_LGBM
        7258             93.000    308              162          4.240          17.290    0.710             0.730     0.715
        6751             86.000    237              107          3.510          22.470    0.761             0.827     0.811
        6542             84.000    768              358         11.740           6.930    0.756             0.841     0.827
        5815             74.000    300              189          5.160          17.750    0.685             0.776     0.752
        6186             79.000    791              358         12.790           6.730    0.622             0.678     0.675
        6948             89.000    349              156          5.020          15.260    0.694             0.738     0.730
        6443             82.000    186              107          2.890          28.630    0.717             0.779     0.738
        

In [8]:
print(Stats)

               n_annotated  n_annotated_share  n_pos  n_scaffolds_pos  \
task                                                                    
NR-AR                 7258               93.0    308              162   
NR-AR-LBD             6751               86.0    237              107   
NR-AhR                6542               84.0    768              358   
NR-ER                 5815               74.0    300              189   
NR-ER-LBD             6186               79.0    791              358   
NR-PPAR-gamma         6948               89.0    349              156   
NR-Aromatase          6443               82.0    186              107   
SR-ARE                5825               74.0    942              439   
SR-ATAD5              7065               90.0    264              144   
SR-HSE                6460               83.0    372              193   
SR-MMP                5804               74.0    918              412   
SR-p53                6767               87.0    42